# BertWordPieceTokenizer

BertWordPieceTokenizer는 구글의 BERT 모델에서 사용되는 WordPiece 알고리즘을 기반으로 한 토크나이저이다. 이는 텍스트를 단순히 공백으로 나누는 것이 아니라, 의미 있는 최소 단위인 서브워드(Subword)로 분리하여 효율적인 텍스트 처리를 돕는다.

BERT 모델을 학습하는 것이 아니라, BERT 계열 모델에서 사용하는 WordPiece 방식의 토크나이저가 텍스트를 어떤 형태로 바꾸는지 확인하는 것이 목적이다.

## 1. 핵심 알고리즘: WordPiece

WordPiece 알고리즘은 단어를 더 작은 단위로 쪼개어 미등록어(Out-of-Vocabulary, OOV) 문제를 줄이는 서브워드 분할 방식이다.

- 자주 등장하는 단어는 하나의 토큰으로 유지될 수 있다.
- 드물게 등장하거나 긴 단어는 여러 개의 서브워드로 분리된다.
- 단어 중간이나 끝에 이어지는 조각에는 `##` 접두어가 붙는다.

예를 들어 다음과 같이 분리될 수 있다.

```text
학습한다 → 학습 + ##한다
가격대비상품괜찮네요 → 가격 + ##대비 + ##상품 + ##괜찮 + ##네요
```

`##`는 해당 토큰이 새로운 단어의 시작이 아니라 앞 토큰에 이어지는 조각임을 나타낸다.

## 2. 주요 특징 및 구성 요소

### 서브워드 접두어 `##`

WordPiece에서는 단어 내부에 이어지는 토큰 앞에 `##`를 붙인다.

```text
편안하구요 → 편안 + ##하 + ##구 + ##요
```

이때 `편안`은 단어의 시작 조각이고, `##하`, `##구`, `##요`는 앞 토큰에 이어지는 조각이다.

### 특수 토큰

BERT 계열 모델에서는 다음과 같은 특수 토큰을 사용한다.

| 토큰 | 의미 | 역할 |
|---|---|---|
| `[PAD]` | Padding | 문장의 길이를 동일하게 맞추기 위해 채우는 토큰 |
| `[UNK]` | Unknown | 어휘 사전에 없는 토큰을 처리하기 위한 토큰 |
| `[CLS]` | Classification | 문장의 시작 위치에 추가되는 토큰 |
| `[SEP]` | Separator | 문장의 끝 또는 두 문장의 구분 위치에 추가되는 토큰 |
| `[MASK]` | Masking | 일부 토큰을 가리고 예측하는 학습에 사용되는 토큰 |

## 3. SentencePiece와 BertWordPieceTokenizer 비교

| 구분 | SentencePiece | BertWordPieceTokenizer |
|---|---|---|
| 대표 표기 | `▁` | `##` |
| 공백 처리 | 공백을 하나의 문자처럼 처리 | 공백이나 구두점 기준으로 먼저 나눈 뒤 처리 |
| 사전 토큰화 | 필수 아님 | 일반적으로 수행 |
| 서브워드 표시 | 단어 시작 여부를 `▁`로 표현 | 단어 내부 조각을 `##`로 표현 |
| 특수 토큰 | 사용자가 지정 가능 | BERT 계열 특수 토큰을 주로 사용 |

두 방식 모두 단어를 더 작은 단위로 나누는 subword tokenizer이다. 다만 공백 처리 방식과 서브워드 표시 방식에 차이가 있다.

In [1]:
# %pip install tokenizer

In [2]:
from tokenizers import BertWordPieceTokenizer
from tokenizers.processors import BertProcessing

## BertWordPieceTokenizer 학습

In [3]:
corpus_file = 'data/female_fashion_reviews.txt'

tokenizer = BertWordPieceTokenizer(
    clean_text = True,              # 텍스트 정제 여부 (공백 정리 등 기본적인 정제 수행)
    handle_chinese_chars = True,          # 한자/중국어 문자를 개별 문자처럼 분리하도록 처리
    strip_accents = False,          # 악센트 제거 여부 (한국어/원문 보존 목적이면 false)
    lowercase=False                 # 소문자 변환 여부 (대문자 보존 목적이면 false)
)

tokenizer.train(
    files = [corpus_file],
    vocab_size = 1000,
    min_frequency = 2,
    special_tokens = [
        '[PAD]','[UNK]','[CLS]','[SEP]','[MASK]'
    ]
)

print('학습 된 vocab_size : ',tokenizer.get_vocab_size())

학습 된 vocab_size :  1000


In [4]:
# 특수 토큰 확인
special_tokens = ['[PAD]','[UNK]','[CLS]','[SEP]','[MASK]']

for token in special_tokens:
    print(token, '->', tokenizer.token_to_id(token))

[PAD] -> 0
[UNK] -> 1
[CLS] -> 2
[SEP] -> 3
[MASK] -> 4


## post-processing 설정
BERT 계열 입력에서는 일반적으로 문장 앞에 '[CLS]', 문장 끝에 '[SEP]' 토큰을 추가한다.
'post-processing'을 설정하면 'encode()'결과에 해당 토큰들이 자동으로 포함된다.

In [5]:
tokenizer.post_processor = BertProcessing(
    # 첫 번째 인자는 SEP 토큰 정보
    ('[SEP]', tokenizer.token_to_id('[SEP]')),
    # 두 번째 인자는 CLS  토큰 정보
    ('[CLS]',tokenizer.token_to_id('[CLS]'))
)

## vocab 및 tokenizer 저장
학습한 WordPiece voacb을 model에 저장한다.

In [6]:
from pathlib import Path

MODEL_DIR = Path('model')
MODEL_DIR.mkdir(exist_ok=True)

vocab_prefix = 'female_fashion_wordpiece'
vocab_file = MODEL_DIR/f'{vocab_prefix}-vocab.txt'
tokenizer_json_file = MODEL_DIR/f'{vocab_prefix}.json'

# vocab 저장
tokenizer.save_model(str(MODEL_DIR),vocab_prefix)

# tokenizer 전체 설정 저장
tokenizer.save(str(tokenizer_json_file))

print('vocab 저장 완료 : ', vocab_file)
print('tokenizer json 저장 완료 : ', tokenizer_json_file)

vocab 저장 완료 :  model\female_fashion_wordpiece-vocab.txt
tokenizer json 저장 완료 :  model\female_fashion_wordpiece.json


## vocab 파일 일부 확인

In [7]:
with open(vocab_file, 'r', encoding='utf-8') as f:
    vocab_tokens = f.read().splitlines()

print('vocab file 토큰 수 : ', len(vocab_tokens))
vocab_tokens[:30]

vocab file 토큰 수 :  1000


['[PAD]',
 '[UNK]',
 '[CLS]',
 '[SEP]',
 '[MASK]',
 '!',
 '(',
 ')',
 ',',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '?',
 'O',
 '^',
 'b',
 '~',
 'ㅋ',
 'ㅎ',
 'ㅛ',
 'ㅜ',
 'ㅠ']

## 리뷰  문장 토큰화

In [8]:
with open(corpus_file, 'r', encoding='utf-8') as f:
    reviews = [line.strip() for line in f if line.strip()]

for review in reviews[:3]:
    encoded = tokenizer.encode(review)
    print('Text : ', review)
    print('Tokens :', encoded.tokens)
    print('ids : ',encoded.ids)
    print()

Text :  얇고 가벼워요 편한데 색이나 모양도 예쁘고요 얇은 밴딩 바지들은 앞이 좀 민망할 때가 있는데 포켓이 있어 앞 라인도 안 민망하고 숨겨진 밴딩때문에 정말 편합니다 색감도 예쁘고 깔끔해서 좋아요 시원함과 편안함 때문에 아웃도어룩을 많이 입었는데 이번 여름은 진 세 장 번갈아 입으며 살고 있어요.
Tokens : ['[CLS]', '얇', '##고', '가', '##벼', '##워', '##요', '편', '##한데', '색', '##이나', '모', '##양', '##도', '예쁘고', '##요', '얇', '##은', '밴', '##딩', '바지', '##들', '##은', '앞', '##이', '좀', '민', '##망', '##할', '때', '##가', '있', '##는데', '포', '##켓', '##이', '있어', '앞', '라', '##인', '##도', '안', '민', '##망', '##하고', '숨', '##겨', '##진', '밴', '##딩', '##때', '##문', '##에', '정말', '편', '##합니다', '색', '##감', '##도', '예쁘고', '깔', '##끔', '##해서', '좋아요', '시원', '##함', '##과', '편안', '##함', '때', '##문', '##에', '아', '##웃', '##도', '##어', '##룩', '##을', '많', '##이', '입', '##었', '##는데', '이', '##번', '여름', '##은', '진', '세', '장', '번', '##갈', '##아', '입으', '##며', '살', '##고', '있어요', '.', '[SEP]']
ids :  [2, 317, 521, 31, 638, 519, 500, 458, 957, 268, 981, 208, 559, 546, 966, 500, 317, 555, 230, 747, 881, 741, 555, 311, 504, 381, 218, 718, 602, 150, 498, 363, 877, 460, 811,

## decode 확인

In [9]:
decoded = tokenizer.decode(encoded.ids)
print(decoded)

몇년 전 OOO 쿨데님 사셨는데 이번에 새로 샀습니다. 두가지 색상이 비슷한게 아쉽지만 소재가 마음에 든다고 하네요. 얇고 스판끼도 많아서 편하게 입을 스타일이래요.


## 알 수 없는 토큰 확인
WordPiece는 vocab에 없는 조각을 처리할때 '[UNK]' 토큰을 사용할 수 있다.

In [11]:
unknown_texts = [
    '처음보는초특급신상블라우스입니다.',
    '이 옷은 👌'
]

for text in unknown_texts:
    encoded = tokenizer.encode(text)
    print('Text : ', text)
    print('Tokens :', encoded.tokens)
    print('ids : ',encoded.ids)
    print()

Text :  처음보는초특급신상블라우스입니다.
Tokens : ['[CLS]', '[UNK]', '.', '[SEP]']
ids :  [2, 1, 9, 3]

Text :  이 옷은 👌
Tokens : ['[CLS]', '이', '옷', '##은', '[UNK]', '[SEP]']
ids :  [2, 358, 334, 555, 1, 3]



## padding과 truncaton 설정
모델에 여러 문장을 한번에 입력하려면 길이를 맞춰 주어야 한다.

In [14]:
max_len = 50

# 문장이 max_len보다 길면 뒤쪽 토큰을 잘라낸다.
tokenizer.enable_truncation(max_length=max_len)

# 문장이 max_len 보다 짧으면 ['PAD'] 토큰으로 채워 길이를 맞춘다.
tokenizer.enable_padding(
    length = max_len,
    pad_id = tokenizer.token_to_id('[PAD]'),
    pad_token = '[PAD]'
)

encoded = tokenizer.encode(reviews[0])
print('tokens : ', encoded.tokens)
print('ids : ', encoded.ids)
print('attenion mask : ', encoded.attention_mask)

encoded = tokenizer.encode(reviews[1])
print('tokens : ', encoded.tokens)
print('ids : ', encoded.ids)
print('attenion mask : ', encoded.attention_mask)

tokens :  ['[CLS]', '얇', '##고', '가', '##벼', '##워', '##요', '편', '##한데', '색', '##이나', '모', '##양', '##도', '예쁘고', '##요', '얇', '##은', '밴', '##딩', '바지', '##들', '##은', '앞', '##이', '좀', '민', '##망', '##할', '때', '##가', '있', '##는데', '포', '##켓', '##이', '있어', '앞', '라', '##인', '##도', '안', '민', '##망', '##하고', '숨', '##겨', '##진', '밴', '[SEP]']
ids :  [2, 317, 521, 31, 638, 519, 500, 458, 957, 268, 981, 208, 559, 546, 966, 500, 317, 555, 230, 747, 881, 741, 555, 311, 504, 381, 218, 718, 602, 150, 498, 363, 877, 460, 811, 504, 932, 311, 159, 661, 546, 307, 218, 718, 879, 285, 867, 625, 230, 3]
attenion mask :  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
tokens :  ['[CLS]', '가격', '##대', '##비', '##상', '##품', '##괜', '##찮', '##네요', '.', '착용감', '##도', '##편', '##안', '##하', '##구요', '.', '잘입', '##겠', '##습니다', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PA

## 여러 문장 한 번에 인코딩
'encode_batch()'를 사용하면 여러 문장을 한 번에 인코딩 할 수 있다.

In [16]:
batch = tokenizer.encode_batch(reviews[:3])

for encoded in batch:
    print('tokens : ', encoded.tokens)
    print('ids : ', encoded.ids)
    print('attenion mask : ', encoded.attention_mask)
    print()

tokens :  ['[CLS]', '얇', '##고', '가', '##벼', '##워', '##요', '편', '##한데', '색', '##이나', '모', '##양', '##도', '예쁘고', '##요', '얇', '##은', '밴', '##딩', '바지', '##들', '##은', '앞', '##이', '좀', '민', '##망', '##할', '때', '##가', '있', '##는데', '포', '##켓', '##이', '있어', '앞', '라', '##인', '##도', '안', '민', '##망', '##하고', '숨', '##겨', '##진', '밴', '[SEP]']
ids :  [2, 317, 521, 31, 638, 519, 500, 458, 957, 268, 981, 208, 559, 546, 966, 500, 317, 555, 230, 747, 881, 741, 555, 311, 504, 381, 218, 718, 602, 150, 498, 363, 877, 460, 811, 504, 932, 311, 159, 661, 546, 307, 218, 718, 879, 285, 867, 625, 230, 3]
attenion mask :  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

tokens :  ['[CLS]', '가격', '##대', '##비', '##상', '##품', '##괜', '##찮', '##네요', '.', '착용감', '##도', '##편', '##안', '##하', '##구요', '.', '잘입', '##겠', '##습니다', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[P

## PyTorch Tensor로 변환
BERT 계열 모델에 입력할 때는 보통 다음과 같은 텐서 형태를 사용한다.

- input_ids: 토큰 id
- attention_mask: 실제 토큰과 패딩 토큰 구분 정보

In [17]:
import torch

# batch 안에는 Encoding 객체들이 들어있고, 각 객체의 ids를 모아서 2차원 리스트로 만든다.
# (batch_size, seq_len) 형태가 된다.
input_ids = torch.tensor(
    [encoded.ids for encoded in batch],  # 각 문장의 토큰 id 리스트
    dtype=torch.long                     
)

# attention_mask도 동일하게 2차원 텐서로 만든다.
attention_mask = torch.tensor(
    [encoded.attention_mask for encoded in batch],
    dtype=torch.long
)

# 텐서의 shape 확인
# (문장 개수, 토큰 길이)
print('input_ids shape:', input_ids.shape)
print('attention_mask shape:', attention_mask.shape)
print()

# 실제 토큰 id 값 출력
print(input_ids)
print()

# attention mask 출력 (패딩 위치 확인 가능)
print(attention_mask)

input_ids shape: torch.Size([3, 50])
attention_mask shape: torch.Size([3, 50])

tensor([[  2, 317, 521,  31, 638, 519, 500, 458, 957, 268, 981, 208, 559, 546,
         966, 500, 317, 555, 230, 747, 881, 741, 555, 311, 504, 381, 218, 718,
         602, 150, 498, 363, 877, 460, 811, 504, 932, 311, 159, 661, 546, 307,
         218, 718, 879, 285, 867, 625, 230,   3],
        [  2, 882, 580, 581, 582, 547, 583, 533, 872,   9, 939, 546, 544, 526,
         530, 960,   9, 950, 593, 895,   9,   3,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [  2, 207, 543, 371,  21, 667, 667, 432, 564, 792, 262, 859, 877, 358,
         760, 523, 267, 525, 265, 895,   9, 135, 498, 496, 912, 504, 253, 552,
         527, 553, 306, 699, 894, 278, 637, 498, 994, 140, 959, 467, 872,   9,
         317, 521, 890, 549, 546, 194, 888,   3]])

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

## Dataset과 DataLoader 연결
실제 딥러닝 학습에서는 텍스트를 매번 같은 방식으로 인코딩하여 batch 단위로 꺼내 쓰는 구조가 필요하다.

In [18]:
from torch.utils.data import Dataset, DataLoader

class ReviewWordPieceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=50):
        self.texts = texts          
        self.tokenizer = tokenizer  
        self.max_len = max_len      

        # max_len보다 긴 문장은 잘라낸다.
        self.tokenizer.enable_truncation(max_length=max_len)

        # max_len보다 짧은 문장은 [PAD] 토큰으로 채운다.
        self.tokenizer.enable_padding(
            length=max_len,                            
            pad_id=self.tokenizer.token_to_id('[PAD]'), 
            pad_token='[PAD]'                          
        )

    def __len__(self):
        # Dataset의 전체 샘플 개수 반환
        return len(self.texts)

    def __getitem__(self, idx):
        # idx번째 리뷰 문장을 토큰화
        encoded = self.tokenizer.encode(self.texts[idx])

        # 토큰 id를 PyTorch 텐서로 변환
        input_ids = torch.tensor(encoded.ids, dtype=torch.long)

        # attention mask를 PyTorch 텐서로 변환
        attention_mask = torch.tensor(encoded.attention_mask, dtype=torch.long)

        # 모델 입력에 사용할 값을 딕셔너리 형태로 반환
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }

In [19]:
# 리뷰 리스트와 tokenizer를 사용해 Dataset 객체 생성
dataset = ReviewWordPieceDataset(reviews, tokenizer, max_len=50)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

batch = next(iter(dataloader))

# input_ids의 shape 확인
# 형태: (batch_size, max_len)
print('input_ids shape:', batch['input_ids'].shape)

# attention_mask의 shape 확인
# 형태: (batch_size, max_len)
print('attention_mask shape:', batch['attention_mask'].shape)
print()

print(batch['input_ids'])
print()

print(batch['attention_mask'])

input_ids shape: torch.Size([2, 50])
attention_mask shape: torch.Size([2, 50])

tensor([[  2, 281, 628, 537, 435, 942, 307, 897, 889, 383, 984, 514, 546, 977,
         504, 493, 504, 932, 514, 878, 141, 596, 765, 496, 546, 308, 499, 212,
         759, 958, 893, 284, 363, 593, 876,   9, 334, 555, 381, 922,  86, 693,
         561,  26,   3,   0,   0,   0,   0,   0],
        [  2, 898, 872, 468, 987, 695, 553, 618, 546, 134, 786, 989, 955, 497,
         278, 637, 498, 878, 300, 565, 985, 992, 509, 569, 625, 208, 630, 593,
         876,   3,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0]])

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0,
         0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    